# Load libraries

In [ ]:
import os
import gc
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import seaborn as sns

import skopt
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import minmax_scale
from sklearn.preprocessing import MinMaxScaler

from sklearn import preprocessing, model_selection
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import QuantileTransformer
from sklearn.model_selection import GroupKFold
from scipy.optimize import minimize

from sklearn.neighbors import NearestNeighbors,KNeighborsClassifier, NeighborhoodComponentsAnalysis, KNeighborsRegressor, LocalOutlierFactor

import xgboost as xgb
from xgboost import XGBRegressor
import lightgbm as lgb

import optiver_test_dataset_build as opt_test
import optiver_test_feature_creation as opt
import optiver_feature_creation2 as opt2
import optiver_feature_creation3 as opt3

# Custom functions

In [ ]:
def pickle_dump(path, saveobj):
    import pickle
    filehandler = open(path,"wb")
    pickle.dump(saveobj,filehandler)
    print("File pickled")
    filehandler.close()

In [ ]:
def pickle_load(path):
    import pickle
    file = open(path,'rb')
    loadobj = pickle.load(file)
    file.close()
    return loadobj

In [ ]:
def log_return(list_stock_prices):
    return np.log(list_stock_prices).diff()

In [ ]:
def realized_volatility(series_log_return):
    return np.sqrt(np.sum(series_log_return**2))

In [ ]:
def rmspe(y_true, y_pred):
    return  (np.sqrt(np.mean(np.square((y_true - y_pred) / y_true))))

In [ ]:
def feval_rmspe(y_pred, lgb_train):
    y_true = lgb_train.get_label()
    return 'RMSPE', rmspe(y_true, y_pred), False

# Read in data

In [ ]:
train_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/train.csv")
test_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/test.csv")
submit_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/sample_submission.csv")

In [ ]:
display(test_df.head(2))
display(submit_df.head(2))

# Build test dataset

## NRU & RU

In [ ]:
stockLst = test_df['stock_id'].unique().tolist()

results = []
results_500 = []
results_400 = []
results_300 = []
results_200 = []
results_100 = []

for stock in tqdm(stockLst):
    temp = opt.nru_features(stock)
    
    results.append(opt.ru_features(temp, stock=stock))
    results_500.append(opt.ru_features(temp, stock=stock, since_second=500))
    results_400.append(opt.ru_features(temp, stock=stock, since_second=400))
    results_300.append(opt.ru_features(temp, stock=stock, since_second=300))
#     results_200.append(opt.ru_features(temp, stock=stock, since_second=200))
#     results_100.append(opt.ru_features(temp, stock=stock, since_second=100))
   
    del temp
    _ = gc.collect()

In [ ]:
opt_test_df = pd.concat(results)
opt_test_df = opt_test_df.reset_index(drop=False)

del results
_ = gc.collect()

opt_test_df_500 = pd.concat(results_500)
opt_test_df_500 = opt_test_df_500.reset_index(drop=False)

del results_500
_ = gc.collect()

opt_test_df_400 = pd.concat(results_400)
opt_test_df_400 = opt_test_df_400.reset_index(drop=False)

del results_400
_ = gc.collect()

opt_test_df_300 = pd.concat(results_300)
opt_test_df_300 = opt_test_df_300.reset_index(drop=False)

del results_300
_ = gc.collect()

## Illidan

In [ ]:
opt_illi_df = opt_test.build_dataset(opt_test_df, opt_test_df_500, opt_test_df_400, opt_test_df_300)

del opt_test_df, opt_test_df_500, opt_test_df_400, opt_test_df_300
_ = gc.collect()

illiCols = pickle_load("/kaggle/input/optiver-training-data/illiCols.pkl")

opt_illi_df = opt_illi_df[['stock_id','time_id'] + illiCols]

print(opt_illi_df.shape)
opt_illi_df.head()

## Public data small

In [ ]:
train, public1_df = opt2.read_train_test()

del train
_ = gc.collect()

# data directory
data_dir = '../input/optiver-realized-volatility-prediction/'

In [ ]:
# Get unique stock ids 
test_stock_ids = public1_df['stock_id'].unique()
# Preprocess them using Parallel and our single stock id functions
test_ = opt2.preprocessor(test_stock_ids, is_train = False)
public1_df = public1_df.merge(test_, on = ['row_id'], how = 'left')
gc.collect()

In [ ]:
public1_df['trade_size_tau'] = np.sqrt(1/public1_df['trade_order_count_sum'])
    
for w in range(150, 600, 150):
    public1_df['trade_size_tau_150win_'+str(w)] = np.sqrt(1/public1_df['trade_order_count_sum_150win_'+str(w)])

public1_df = public1_df.reset_index(drop=True)

In [ ]:
opt_illi_df = pd.merge(opt_illi_df, public1_df, on=['time_id','stock_id'], how="left")
opt_illi_df.shape

In [ ]:
del public1_df
_ = gc.collect()

## Public data big

In [ ]:
path_submissions = '/'

target_name = 'target'
scores_folds = {}

# data directory
data_dir = '../input/optiver-realized-volatility-prediction/'


# Read train and test
train =pd.read_pickle("/kaggle/input/optiver006/train.pkl")
public2_df = opt3.read_train_test()

In [ ]:
# Get unique stock ids 
test_stock_ids = public2_df['stock_id'].unique()
# Preprocess them using Parallel and our single stock id functions
test_ = opt3.preprocessor(test_stock_ids, is_train = False)
public2_df = public2_df.merge(test_, on = ['row_id'], how = 'left')

# Get group stats of time_id and stock_id
#train = get_time_stock(train)
public2_df = opt3.get_time_stock(public2_df)

train1=train
test1=public2_df

In [ ]:
# replace by order sum (tau)
train['size_tau'] = np.sqrt( 1/ train['trade_seconds_in_bucket_count_unique'] )
public2_df['size_tau'] = np.sqrt( 1/ public2_df['trade_seconds_in_bucket_count_unique'] )

train['size_tau_400'] = np.sqrt( 1/ train['trade_seconds_in_bucket_count_unique_400'] )
public2_df['size_tau_400'] = np.sqrt( 1/ public2_df['trade_seconds_in_bucket_count_unique_400'] )

train['size_tau_300'] = np.sqrt( 1/ train['trade_seconds_in_bucket_count_unique_300'] )
public2_df['size_tau_300'] = np.sqrt( 1/ public2_df['trade_seconds_in_bucket_count_unique_300'] )

train['size_tau_200'] = np.sqrt( 1/ train['trade_seconds_in_bucket_count_unique_200'] )
public2_df['size_tau_200'] = np.sqrt( 1/ public2_df['trade_seconds_in_bucket_count_unique_200'] )


train['size_tau2'] = np.sqrt( 1/ train['trade_order_count_sum'] )
public2_df['size_tau2'] = np.sqrt( 1/ public2_df['trade_order_count_sum'] )

train['size_tau2_400'] = np.sqrt( 0.33/ train['trade_order_count_sum'] )
public2_df['size_tau2_400'] = np.sqrt( 0.33/ public2_df['trade_order_count_sum'] )

train['size_tau2_300'] = np.sqrt( 0.5/ train['trade_order_count_sum'] )
public2_df['size_tau2_300'] = np.sqrt( 0.5/ public2_df['trade_order_count_sum'] )

train['size_tau2_200'] = np.sqrt( 0.66/ train['trade_order_count_sum'] )
public2_df['size_tau2_200'] = np.sqrt( 0.66/ public2_df['trade_order_count_sum'] )

# delta tau
train['size_tau2_d'] = train['size_tau2_400'] - train['size_tau2']
public2_df['size_tau2_d'] = public2_df['size_tau2_400'] - public2_df['size_tau2']

In [ ]:
colNames = [col for col in list(train.columns)
            if col not in {"stock_id", "time_id", "target", "row_id"}]
len(colNames)

In [ ]:
from sklearn.cluster import KMeans
# making agg features

train_p = pd.read_csv('../input/optiver-realized-volatility-prediction/train.csv')
train_p = train_p.pivot(index='time_id', columns='stock_id', values='target')

corr = train_p.corr()

ids = corr.index

kmeans = KMeans(n_clusters=7, random_state=0).fit(corr.values)
print(kmeans.labels_)

l = []
for n in range(7):
    l.append ( [ (x-1) for x in ( (ids+1)*(kmeans.labels_ == n)) if x > 0] )
    

mat = []
matTest = []

n = 0
for ind in l:
    print(ind)
    newDf = train.loc[train['stock_id'].isin(ind) ]
    newDf = newDf.groupby(['time_id']).agg(np.nanmean)
    newDf.loc[:,'stock_id'] = str(n)+'c1'
    mat.append ( newDf )
    
    newDf = public2_df.loc[public2_df['stock_id'].isin(ind) ]    
    newDf = newDf.groupby(['time_id']).agg(np.nanmean)
    newDf.loc[:,'stock_id'] = str(n)+'c1'
    matTest.append ( newDf )
    
    n+=1
    
mat1 = pd.concat(mat).reset_index()
mat1.drop(columns=['target'],inplace=True)

mat2 = pd.concat(matTest).reset_index()

In [ ]:
mat2 = pd.concat([mat2,mat1.loc[mat1.time_id==5]])
mat1 = mat1.pivot(index='time_id', columns='stock_id')
mat1.columns = ["_".join(x) for x in mat1.columns.ravel()]
mat1.reset_index(inplace=True)

mat2 = mat2.pivot(index='time_id', columns='stock_id')
mat2.columns = ["_".join(x) for x in mat2.columns.ravel()]
mat2.reset_index(inplace=True)

In [ ]:
nnn = ['time_id',
     'log_return1_realized_volatility_0c1',
     'log_return1_realized_volatility_1c1',     
     'log_return1_realized_volatility_3c1',
     'log_return1_realized_volatility_4c1',     
     'log_return1_realized_volatility_6c1',
     'total_volume_sum_0c1',
     'total_volume_sum_1c1', 
     'total_volume_sum_3c1',
     'total_volume_sum_4c1', 
     'total_volume_sum_6c1',
     'trade_size_sum_0c1',
     'trade_size_sum_1c1', 
     'trade_size_sum_3c1',
     'trade_size_sum_4c1', 
     'trade_size_sum_6c1',
     'trade_order_count_sum_0c1',
     'trade_order_count_sum_1c1',
     'trade_order_count_sum_3c1',
     'trade_order_count_sum_4c1',
     'trade_order_count_sum_6c1',      
     'price_spread_sum_0c1',
     'price_spread_sum_1c1',
     'price_spread_sum_3c1',
     'price_spread_sum_4c1',
     'price_spread_sum_6c1',   
     'bid_spread_sum_0c1',
     'bid_spread_sum_1c1',
     'bid_spread_sum_3c1',
     'bid_spread_sum_4c1',
     'bid_spread_sum_6c1',       
     'ask_spread_sum_0c1',
     'ask_spread_sum_1c1',
     'ask_spread_sum_3c1',
     'ask_spread_sum_4c1',
     'ask_spread_sum_6c1',   
     'volume_imbalance_sum_0c1',
     'volume_imbalance_sum_1c1',
     'volume_imbalance_sum_3c1',
     'volume_imbalance_sum_4c1',
     'volume_imbalance_sum_6c1',       
     'bid_ask_spread_sum_0c1',
     'bid_ask_spread_sum_1c1',
     'bid_ask_spread_sum_3c1',
     'bid_ask_spread_sum_4c1',
     'bid_ask_spread_sum_6c1',
     'size_tau2_0c1',
     'size_tau2_1c1',
     'size_tau2_3c1',
     'size_tau2_4c1',
     'size_tau2_6c1'] 
train = pd.merge(train,mat1[nnn],how='left',on='time_id')
public2_df = pd.merge(public2_df,mat2[nnn],how='left',on='time_id')

In [ ]:
import gc
del mat1,mat2
gc.collect()

In [ ]:
# add extra features based on highest feature importance

train['total_volume_sum_0c1/trade_size_sum_0c1'] = train['total_volume_sum_0c1'] / train['trade_size_sum_0c1']
public2_df['total_volume_sum_0c1/trade_size_sum_0c1'] = public2_df['total_volume_sum_0c1'] / public2_df['trade_size_sum_0c1']

train['total_volume_sum_3c1/size_tau2_3c1'] = train['total_volume_sum_3c1'] / train['size_tau2_3c1']
public2_df['total_volume_sum_3c1/size_tau2_3c1'] = public2_df['total_volume_sum_3c1'] / public2_df['size_tau2_3c1']

train['total_volume_sum_1c1/size_tau2_1c1'] = train['total_volume_sum_1c1'] / train['size_tau2_1c1']
public2_df['total_volume_sum_1c1/size_tau2_1c1'] = public2_df['total_volume_sum_1c1'] / public2_df['size_tau2_1c1']

train['stock_id/price_spread_sum'] = train['stock_id'] / train['price_spread_sum']
public2_df['stock_id/price_spread_sum'] = public2_df['stock_id'] / public2_df['price_spread_sum']

In [ ]:
public1Cols = pickle_load("/kaggle/input/optiver-training-data/public1Cols.pkl")

test = public2_df.copy()

public2_df = public2_df[['stock_id','time_id'] + public1Cols]

public2_df.shape

In [ ]:
opt_illi_df = pd.merge(opt_illi_df, public2_df, on=['time_id','stock_id'], how="left")
opt_illi_df.shape

In [ ]:
del public2_df
_ = gc.collect()

# Public models

In [ ]:
from sklearn.model_selection import KFold
import lightgbm as lgb

seed0=2021
params0 = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
    'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'verbose': -1}
seed1=42
params1 = {
        'learning_rate': 0.1,        
        'lambda_l1': 2,
        'lambda_l2': 7,
        'num_leaves': 800,
        'min_sum_hessian_in_leaf': 20,
        'feature_fraction': 0.8,
        'feature_fraction_bynode': 0.8,
        'bagging_fraction': 0.9,
        'bagging_freq': 42,
        'min_data_in_leaf': 700,
        'max_depth': 4,
        'categorical_column':[0],
        'seed': seed1,
        'feature_fraction_seed': seed1,
        'bagging_seed': seed1,
        'drop_seed': seed1,
        'data_random_seed': seed1,
        'objective': 'rmse',
        'boosting': 'gbdt',
        'verbosity': -1,
        'n_jobs':-1,
    }
# Function to early stop with root mean squared percentage error
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

def feval_rmspe(y_pred, lgb_train):
    y_true = lgb_train.get_label()
    return 'RMSPE', rmspe(y_true, y_pred), False

def train_and_evaluate_lgb(train, test, params):
    # Hyperparammeters (just basic)
    
    features = [col for col in train.columns if col not in {"time_id", "target", "row_id"}]
    y = train['target']
    # Create out of folds array
    oof_predictions = np.zeros(train.shape[0])
    # Create test array to store predictions
    test_predictions = np.zeros(test.shape[0])
    # Create a KFold object
    kfold = KFold(n_splits = 5, random_state = 2021, shuffle = True)
    # Iterate through each fold
    for fold, (trn_ind, val_ind) in enumerate(kfold.split(train)):
        print(f'Training fold {fold + 1}')
        x_train, x_val = train.iloc[trn_ind], train.iloc[val_ind]
        y_train, y_val = y.iloc[trn_ind], y.iloc[val_ind]
        # Root mean squared percentage error weights
        train_weights = 1 / np.square(y_train)
        val_weights = 1 / np.square(y_val)
        train_dataset = lgb.Dataset(x_train[features], y_train, weight = train_weights)
        val_dataset = lgb.Dataset(x_val[features], y_val, weight = val_weights)
        model = lgb.train(params = params,
                          num_boost_round=1000,
                          train_set = train_dataset, 
                          valid_sets = [train_dataset, val_dataset], 
                          verbose_eval = 250,
                          early_stopping_rounds=50,
                          feval = feval_rmspe)
        # Add predictions to the out of folds array
        oof_predictions[val_ind] = model.predict(x_val[features])
        # Predict the test set
        test_predictions += model.predict(test[features]) / 5
    rmspe_score = rmspe(y, oof_predictions)
    print(f'Our out of folds RMSPE is {rmspe_score}')
    lgb.plot_importance(model,max_num_features=20)
    # Return test predictions
    return oof_predictions, test_predictions
# Traing and evaluate
oof_lgb1, predictions_lgb1= train_and_evaluate_lgb(train, test,params0)
#test['target'] = predictions_lgb
#test[['row_id', 'target']].to_csv('submission.csv',index = False)

In [ ]:
from sklearn.model_selection import KFold
import lightgbm as lgb

seed0=1111
params111 = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
    'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'verbose': -1}
seed1=42
params1 = {
        'learning_rate': 0.1,        
        'lambda_l1': 2,
        'lambda_l2': 7,
        'num_leaves': 800,
        'min_sum_hessian_in_leaf': 20,
        'feature_fraction': 0.8,
        'feature_fraction_bynode': 0.8,
        'bagging_fraction': 0.9,
        'bagging_freq': 42,
        'min_data_in_leaf': 700,
        'max_depth': 4,
        'categorical_column':[0],
        'seed': seed1,
        'feature_fraction_seed': seed1,
        'bagging_seed': seed1,
        'drop_seed': seed1,
        'data_random_seed': seed1,
        'objective': 'rmse',
        'boosting': 'gbdt',
        'verbosity': -1,
        'n_jobs':-1,
    }
# Function to early stop with root mean squared percentage error
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

def feval_rmspe(y_pred, lgb_train):
    y_true = lgb_train.get_label()
    return 'RMSPE', rmspe(y_true, y_pred), False

def train_and_evaluate_lgb(train, test, params):
    # Hyperparammeters (just basic)
    
    features = [col for col in train.columns if col not in {"time_id", "target", "row_id"}]
    y = train['target']
    # Create out of folds array
    oof_predictions = np.zeros(train.shape[0])
    # Create test array to store predictions
    test_predictions = np.zeros(test.shape[0])
    # Create a KFold object
    kfold = KFold(n_splits = 5, random_state = 1111, shuffle = True)
    # Iterate through each fold
    for fold, (trn_ind, val_ind) in enumerate(kfold.split(train)):
        print(f'Training fold {fold + 1}')
        x_train, x_val = train.iloc[trn_ind], train.iloc[val_ind]
        y_train, y_val = y.iloc[trn_ind], y.iloc[val_ind]
        # Root mean squared percentage error weights
        train_weights = 1 / np.square(y_train)
        val_weights = 1 / np.square(y_val)
        train_dataset = lgb.Dataset(x_train[features], y_train, weight = train_weights)
        val_dataset = lgb.Dataset(x_val[features], y_val, weight = val_weights)
        model = lgb.train(params = params,
                          num_boost_round=1200,
                          train_set = train_dataset, 
                          valid_sets = [train_dataset, val_dataset], 
                          verbose_eval = 250,
                          early_stopping_rounds=50,
                          feval = feval_rmspe)
        # Add predictions to the out of folds array
        oof_predictions[val_ind] = model.predict(x_val[features])
        # Predict the test set
        test_predictions += model.predict(test[features]) / 5
    rmspe_score = rmspe(y, oof_predictions)
    print(f'Our out of folds RMSPE is {rmspe_score}')
    lgb.plot_importance(model,max_num_features=20)
    # Return test predictions
    return oof_predictions, test_predictions
# Traing and evaluate
oof_lgb2, predictions_lgb2= train_and_evaluate_lgb(train, test,params111)
#test['target'] = predictions_lgb
#test[['row_id', 'target']].to_csv('submission.csv',index = False)

## NN

In [ ]:
from numpy.random import seed
seed(42)
import tensorflow as tf
tf.random.set_seed(42)
from tensorflow import keras
import numpy as np
from keras import backend as K
def root_mean_squared_per_error(y_true, y_pred):
         return K.sqrt(K.mean(K.square( (y_true - y_pred)/ y_true )))
    
es = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=20, verbose=0,
    mode='min',restore_best_weights=True)

plateau = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=7, verbose=0,
    mode='min')

In [ ]:
#  kfold based on the knn++ algorithm

out_train = pd.read_csv('../input/optiver-realized-volatility-prediction/train.csv')
out_train = out_train.pivot(index='time_id', columns='stock_id', values='target')

#out_train[out_train.isna().any(axis=1)]
out_train = out_train.fillna(out_train.mean())
out_train.head()

# code to add the just the read data after first execution

# data separation based on knn ++
nfolds = 5 # number of folds
index = []
totDist = []
values = []
# generates a matriz with the values of 
mat = out_train.values

scaler = MinMaxScaler(feature_range=(-1, 1))
mat = scaler.fit_transform(mat)

nind = int(mat.shape[0]/nfolds) # number of individuals

# adds index in the last column
mat = np.c_[mat,np.arange(mat.shape[0])]


lineNumber = np.random.choice(np.array(mat.shape[0]), size=nfolds, replace=False)

lineNumber = np.sort(lineNumber)[::-1]

for n in range(nfolds):
    totDist.append(np.zeros(mat.shape[0]-nfolds))

# saves index
for n in range(nfolds):
    
    values.append([lineNumber[n]])    


s=[]
for n in range(nfolds):
    s.append(mat[lineNumber[n],:])
    
    mat = np.delete(mat, obj=lineNumber[n], axis=0)

for n in range(nind-1):    

    luck = np.random.uniform(0,1,nfolds)
    
    for cycle in range(nfolds):
         # saves the values of index           

        s[cycle] = np.matlib.repmat(s[cycle], mat.shape[0], 1)

        sumDist = np.sum( (mat[:,:-1] - s[cycle][:,:-1])**2 , axis=1)   
        totDist[cycle] += sumDist        
                
        # probabilities
        f = totDist[cycle]/np.sum(totDist[cycle]) # normalizing the totdist
        j = 0
        kn = 0
        for val in f:
            j += val        
            if (j > luck[cycle]): # the column was selected
                break
            kn +=1
        lineNumber[cycle] = kn
        
        # delete line of the value added    
        for n_iter in range(nfolds):
            
            totDist[n_iter] = np.delete(totDist[n_iter],obj=lineNumber[cycle], axis=0)
            j= 0
        
        s[cycle] = mat[lineNumber[cycle],:]
        values[cycle].append(int(mat[lineNumber[cycle],-1]))
        mat = np.delete(mat, obj=lineNumber[cycle], axis=0)


for n_mod in range(nfolds):
    values[n_mod] = out_train.index[values[n_mod]]

In [ ]:
#colNames.remove('row_id')
train.replace([np.inf, -np.inf], np.nan,inplace=True)
test.replace([np.inf, -np.inf], np.nan,inplace=True)
qt_train = []
train_nn=train[colNames].copy()
test_nn=test[colNames].copy()
for col in colNames:
    #print(col)
    qt = QuantileTransformer(random_state=21,n_quantiles=2000, output_distribution='normal')
    train_nn[col] = qt.fit_transform(train_nn[[col]])
    test_nn[col] = qt.transform(test_nn[[col]])    
    qt_train.append(qt)

In [ ]:
train_nn[['stock_id','time_id','target']]=train[['stock_id','time_id','target']]
test_nn[['stock_id','time_id']]=test[['stock_id','time_id']]

In [ ]:
# making agg features
from sklearn.cluster import KMeans
train_p = pd.read_csv('../input/optiver-realized-volatility-prediction/train.csv')
train_p = train_p.pivot(index='time_id', columns='stock_id', values='target')

corr = train_p.corr()

ids = corr.index

kmeans = KMeans(n_clusters=7, random_state=0).fit(corr.values)
print(kmeans.labels_)

l = []
for n in range(7):
    l.append ( [ (x-1) for x in ( (ids+1)*(kmeans.labels_ == n)) if x > 0] )
    

mat = []
matTest = []

n = 0
for ind in l:
    print(ind)
    newDf = train_nn.loc[train_nn['stock_id'].isin(ind) ]
    newDf = newDf.groupby(['time_id']).agg(np.nanmean)
    newDf.loc[:,'stock_id'] = str(n)+'c1'
    mat.append ( newDf )
    
    newDf = test_nn.loc[test_nn['stock_id'].isin(ind) ]    
    newDf = newDf.groupby(['time_id']).agg(np.nanmean)
    newDf.loc[:,'stock_id'] = str(n)+'c1'
    matTest.append ( newDf )
    
    n+=1
    
mat1 = pd.concat(mat).reset_index()
mat1.drop(columns=['target'],inplace=True)

mat2 = pd.concat(matTest).reset_index()
mat2 = pd.concat([mat2,mat1.loc[mat1.time_id==5]])

In [ ]:
nnn = ['time_id',
     'log_return1_realized_volatility_0c1',
     'log_return1_realized_volatility_1c1',     
     'log_return1_realized_volatility_3c1',
     'log_return1_realized_volatility_4c1',     
     'log_return1_realized_volatility_6c1',
     'total_volume_sum_0c1',
     'total_volume_sum_1c1', 
     'total_volume_sum_3c1',
     'total_volume_sum_4c1', 
     'total_volume_sum_6c1',
     'trade_size_sum_0c1',
     'trade_size_sum_1c1', 
     'trade_size_sum_3c1',
     'trade_size_sum_4c1', 
     'trade_size_sum_6c1',
     'trade_order_count_sum_0c1',
     'trade_order_count_sum_1c1',
     'trade_order_count_sum_3c1',
     'trade_order_count_sum_4c1',
     'trade_order_count_sum_6c1',      
     'price_spread_sum_0c1',
     'price_spread_sum_1c1',
     'price_spread_sum_3c1',
     'price_spread_sum_4c1',
     'price_spread_sum_6c1',   
     'bid_spread_sum_0c1',
     'bid_spread_sum_1c1',
     'bid_spread_sum_3c1',
     'bid_spread_sum_4c1',
     'bid_spread_sum_6c1',       
     'ask_spread_sum_0c1',
     'ask_spread_sum_1c1',
     'ask_spread_sum_3c1',
     'ask_spread_sum_4c1',
     'ask_spread_sum_6c1',   
     'volume_imbalance_sum_0c1',
     'volume_imbalance_sum_1c1',
     'volume_imbalance_sum_3c1',
     'volume_imbalance_sum_4c1',
     'volume_imbalance_sum_6c1',       
     'bid_ask_spread_sum_0c1',
     'bid_ask_spread_sum_1c1',
     'bid_ask_spread_sum_3c1',
     'bid_ask_spread_sum_4c1',
     'bid_ask_spread_sum_6c1',
     'size_tau2_0c1',
     'size_tau2_1c1',
     'size_tau2_3c1',
     'size_tau2_4c1',
     'size_tau2_6c1'] 

In [ ]:
mat1 = mat1.pivot(index='time_id', columns='stock_id')
mat1.columns = ["_".join(x) for x in mat1.columns.ravel()]
mat1.reset_index(inplace=True)

mat2 = mat2.pivot(index='time_id', columns='stock_id')
mat2.columns = ["_".join(x) for x in mat2.columns.ravel()]
mat2.reset_index(inplace=True)

In [ ]:
import gc
train_nn = pd.merge(train_nn,mat1[nnn],how='left',on='time_id')
test_nn = pd.merge(test_nn,mat2[nnn],how='left',on='time_id')

train1=train_nn
test1=test_nn
del mat1,mat2
del train,test
gc.collect()

In [ ]:
#https://bignerdranch.com/blog/implementing-swish-activation-function-in-keras/
from keras.backend import sigmoid
def swish(x, beta = 1):
    return (x * sigmoid(beta * x))

from keras.utils.generic_utils import get_custom_objects
from keras.layers import Activation
get_custom_objects().update({'swish': Activation(swish)})

hidden_units = (128,64,32)
stock_embedding_size = 24

cat_data = train_nn['stock_id']

def base_model():
    
    # Each instance will consist of two inputs: a single user id, and a single movie id
    stock_id_input = keras.Input(shape=(1,), name='stock_id')
    num_input = keras.Input(shape=(244,), name='num_data')


    #embedding, flatenning and concatenating
    stock_embedded = keras.layers.Embedding(max(cat_data)+1, stock_embedding_size, 
                                           input_length=1, name='stock_embedding')(stock_id_input)
    stock_flattened = keras.layers.Flatten()(stock_embedded)
    out = keras.layers.Concatenate()([stock_flattened, num_input])
    
    # Add one or more hidden layers
    for n_hidden in hidden_units:

        out = keras.layers.Dense(n_hidden, activation='swish')(out)
        

    #out = keras.layers.Concatenate()([out, num_input])

    # A single output: our predicted rating
    out = keras.layers.Dense(1, activation='linear', name='prediction')(out)
    
    model = keras.Model(
    inputs = [stock_id_input, num_input],
    outputs = out,
    )
    
    return model

In [ ]:
# Function to calculate the root mean squared percentage error
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(np.square((y_true - y_pred) / y_true)))

# Function to early stop with root mean squared percentage error
def feval_rmspe(y_pred, lgb_train):
    y_true = lgb_train.get_label()
    return 'RMSPE', rmspe(y_true, y_pred), False

In [ ]:
target_name='target'
scores_folds = {}
model_name = 'NN'
pred_name = 'pred_{}'.format(model_name)

n_folds = 5
kf = model_selection.KFold(n_splits=n_folds, shuffle=True, random_state=2020)
scores_folds[model_name] = []
counter = 1

features_to_consider = list(train_nn)

features_to_consider.remove('time_id')
features_to_consider.remove('target')
try:
    features_to_consider.remove('pred_NN')
except:
    pass


train_nn[features_to_consider] = train_nn[features_to_consider].fillna(train_nn[features_to_consider].mean())
test_nn[features_to_consider] = test_nn[features_to_consider].fillna(train_nn[features_to_consider].mean())

train_nn[pred_name] = 0
test_nn[target_name] = 0
test_predictions_nn = np.zeros(test_nn.shape[0])
# Create out of folds array
oof_nn1 = np.zeros(train_df.shape[0])

for n_count in range(n_folds):
    print('CV {}/{}'.format(counter, n_folds))
    
    indexes = np.arange(nfolds).astype(int)    
    indexes = np.delete(indexes,obj=n_count, axis=0) 
    
    indexes = np.r_[values[indexes[0]],values[indexes[1]],values[indexes[2]],values[indexes[3]]]
    
    X_train = train_nn.loc[train_nn.time_id.isin(indexes), features_to_consider]
    y_train = train_nn.loc[train_nn.time_id.isin(indexes), target_name]
    X_test = train_nn.loc[train_nn.time_id.isin(values[n_count]), features_to_consider]
    y_test = train_nn.loc[train_nn.time_id.isin(values[n_count]), target_name]
    
    val_ind = X_test.index.tolist()
    
    #############################################################################################
    # NN
    #############################################################################################
    
    model = base_model()
    
    model.compile(
        keras.optimizers.Adam(learning_rate=0.006),
        loss=root_mean_squared_per_error
    )
    
    try:
        features_to_consider.remove('stock_id')
    except:
        pass
    
    num_data = X_train[features_to_consider]
    
    scaler = MinMaxScaler(feature_range=(-1, 1))         
    num_data = scaler.fit_transform(num_data.values)    
    
    cat_data = X_train['stock_id']    
    target =  y_train
    
    num_data_test = X_test[features_to_consider]
    num_data_test = scaler.transform(num_data_test.values)
    cat_data_test = X_test['stock_id']

    model.fit([cat_data, num_data], 
              target,               
              batch_size=2048,
              epochs=1000,
              validation_data=([cat_data_test, num_data_test], y_test),
              callbacks=[es, plateau],
              validation_batch_size=len(y_test),
              shuffle=True,
             verbose = 1)

    preds = model.predict([cat_data_test, num_data_test]).reshape(1,-1)[0]
    
    # Add predictions to the out of folds array
    oof_nn1[val_ind] = preds
    
    
    score = round(rmspe(y_true = y_test, y_pred = preds),5)
    print('Fold {} {}: {}'.format(counter, model_name, score))
    scores_folds[model_name].append(score)
    
    tt =scaler.transform(test_nn[features_to_consider].values)
    #test_nn[target_name] += model.predict([test_nn['stock_id'], tt]).reshape(1,-1)[0].clip(0,1e10)
    test_predictions_nn += model.predict([test_nn['stock_id'], tt]).reshape(1,-1)[0].clip(0,1e10)/n_folds
    #test[target_name] += model.predict([test['stock_id'], test[features_to_consider]]).reshape(1,-1)[0].clip(0,1e10)
       
    counter += 1
    features_to_consider.append('stock_id')

In [ ]:
from numpy.random import seed
seed(41)
import tensorflow as tf
tf.random.set_seed(41)
from tensorflow import keras

target_name='target'
scores_folds = {}
model_name = 'NN'
pred_name = 'pred_{}'.format(model_name)

n_folds = 5
kf = model_selection.KFold(n_splits=n_folds, shuffle=True, random_state=2021)
scores_folds[model_name] = []
counter = 1

features_to_consider = list(train1)

features_to_consider.remove('time_id')
features_to_consider.remove('target')
try:
    features_to_consider.remove('pred_NN')
except:
    pass


train1[features_to_consider] = train1[features_to_consider].fillna(train1[features_to_consider].mean())
test1[features_to_consider] = test1[features_to_consider].fillna(train1[features_to_consider].mean())

train1[pred_name] = 0
test1[target_name] = 0
test_predictions_nn1 = np.zeros(test_nn.shape[0])

# Create out of folds array
oof_nn2 = np.zeros(train_df.shape[0])

for n_count in range(n_folds):
    print('CV {}/{}'.format(counter, n_folds))
    
    indexes = np.arange(nfolds).astype(int)    
    indexes = np.delete(indexes,obj=n_count, axis=0) 
    
    indexes = np.r_[values[indexes[0]],values[indexes[1]],values[indexes[2]],values[indexes[3]]]
    
    X_train = train1.loc[train1.time_id.isin(indexes), features_to_consider]
    y_train = train1.loc[train1.time_id.isin(indexes), target_name]
    X_test = train1.loc[train1.time_id.isin(values[n_count]), features_to_consider]
    y_test = train1.loc[train1.time_id.isin(values[n_count]), target_name]
    
    val_ind = X_test.index.tolist()
    
    #############################################################################################
    # NN
    #############################################################################################
    
    model = base_model()
    
    model.compile(
        keras.optimizers.Adam(learning_rate=0.006),
        loss=root_mean_squared_per_error
    )
    
    try:
        features_to_consider.remove('stock_id')
    except:
        pass
    
    num_data = X_train[features_to_consider]
    
    scaler = MinMaxScaler(feature_range=(-1, 1))         
    num_data = scaler.fit_transform(num_data.values)    
    
    cat_data = X_train['stock_id']    
    target =  y_train
    
    num_data_test = X_test[features_to_consider]
    num_data_test = scaler.transform(num_data_test.values)
    cat_data_test = X_test['stock_id']

    model.fit([cat_data, num_data], 
              target,               
              batch_size=2048,
              epochs=1000,
              validation_data=([cat_data_test, num_data_test], y_test),
              callbacks=[es, plateau],
              validation_batch_size=len(y_test),
              shuffle=True,
             verbose = 1)

    preds = model.predict([cat_data_test, num_data_test]).reshape(1,-1)[0]
    # Add predictions to the out of folds array
    oof_nn2[val_ind] = preds
    
    score = round(rmspe(y_true = y_test, y_pred = preds),5)
    print('Fold {} {}: {}'.format(counter, model_name, score))
    scores_folds[model_name].append(score)
    
    tt =scaler.transform(test_nn[features_to_consider].values)
    #test_nn[target_name] += model.predict([test_nn['stock_id'], tt]).reshape(1,-1)[0].clip(0,1e10)
    test_predictions_nn1 += model.predict([test1['stock_id'], tt]).reshape(1,-1)[0].clip(0,1e10)/n_folds
    #test[target_name] += model.predict([test['stock_id'], test[features_to_consider]]).reshape(1,-1)[0].clip(0,1e10)
       
    counter += 1
    features_to_consider.append('stock_id')

# Correlation KO

In [ ]:
corr_cols = ['bid_ask_spread_amax',
 'book_log_return1_realized_volatility',
 'book_log_return1_realized_volatility_150win_300',
 'book_log_return2_realized_volatility',
 'book_log_return2_realized_volatility_150win_300',
 'book_price_spread_sum',
 'log_return1_realized_volatility',
 'log_return1_realized_volatility_200_mean_stock',
 'log_return1_realized_volatility_300',
 'log_return1_realized_volatility_300_mean_stock',
 'log_return1_realized_volatility_400',
 'log_return1_realized_volatility_500',
 'log_return1_realized_volatility_mean_stock',
 'log_return1_realized_volatility_min_stock',
 'log_return1_realized_volatility_min_time',
 'log_return1_realized_volatility_std_time',
 'log_return2_realized_volatility',
 'log_return2_realized_volatility_200_mean_stock',
 'log_return2_realized_volatility_300',
 'log_return2_realized_volatility_300_mean_stock',
 'log_return2_realized_volatility_400',
 'log_return2_realized_volatility_400_mean_stock',
 'log_return2_realized_volatility_min_stock',
 'log_return2_realized_volatility_min_time',
 'log_return2_realized_volatility_std_time',
 'log_return3_realized_volatility',
 'log_return4_realized_volatility',
 'price_spread2_amax',
 'price_spread_amax',
 'price_spread_sum',
 'trade_amount_amax',
 'trade_log_return_realized_volatility_200_mean_stock',
 'trade_log_return_realized_volatility_300',
 'trade_log_return_realized_volatility_300_mean_stock',
 'trade_log_return_realized_volatility_mean_stock',
 'trade_log_return_realized_volatility_y',
 'trade_size_amax',
 'wap_balance_sum']

In [ ]:
opt_illi_df.drop(corr_cols,axis=1,inplace=True)

print(opt_illi_df.shape)

# LightGBM

In [ ]:
opt_test_df = opt_illi_df.copy()
opt_test_df.shape

In [ ]:
# Load model object
model1 = pickle_load("/kaggle/input/optiver-models/lgbm_illi_ens_fold1.pkl")
model2 = pickle_load("/kaggle/input/optiver-models/lgbm_illi_ens_fold2.pkl")
model3 = pickle_load("/kaggle/input/optiver-models/lgbm_illi_ens_fold3.pkl")
model4 = pickle_load("/kaggle/input/optiver-models/lgbm_illi_ens_fold4.pkl")
model5 = pickle_load("/kaggle/input/optiver-models/lgbm_illi_ens_fold5.pkl")

In [ ]:
trn_x = opt_test_df.drop(columns=['time_id','row_id'])#[modelCols]

# train_dataset = lgb.Dataset(trn_x)

preds_illi = 0.2 * model1.predict(trn_x) + 0.2 * model2.predict(trn_x) + 0.2 * model3.predict(trn_x) + 0.2 * model4.predict(trn_x) + 0.2 * model5.predict(trn_x)

In [ ]:
# W = [7.87564613e-01, 5.55111512e-17, 2.12435387e-01, 0.00000000e+00, 0.00000000e+00]

# W = [0.75820049, 0.01, 0.21179951, 0.01, 0.01]

W = [0.8, 0.2]

W

In [ ]:
# test=pd.read_csv("../input/optiver-realized-volatility-prediction/test.csv")
# a=test_predictions_nn*0.60+predictions_lgb2*0.40
# b=test_predictions_nn1*0.55+preds_illi*0.45
# # c=preds_illi
# test[target_name] = (a+b)/2

# display(test[['row_id', target_name]].head(3))
# test[['row_id', target_name]].to_csv('submission.csv',index = False)
# #test[['row_id', target_name]].to_csv('submission.csv',index = False)
# #kmeans N=5 [0.2101, 0.21399, 0.20923, 0.21398, 0.21175]

In [ ]:
w_nn = 0.25
w_nn1 = 0.25
w_illi = 0.4
w_lgb2 = 0.1

print(w_nn + w_nn1 + w_illi + w_lgb2)

In [ ]:
test=pd.read_csv("../input/optiver-realized-volatility-prediction/test.csv")


# c=preds_illi
test[target_name] = test_predictions_nn*w_nn + test_predictions_nn1*w_nn1 + predictions_lgb2*w_lgb2 + preds_illi*w_illi

display(test[['row_id', target_name]].head(3))
test[['row_id', target_name]].to_csv('submission.csv',index = False)
#test[['row_id', target_name]].to_csv('submission.csv',index = False)
#kmeans N=5 [0.2101, 0.21399, 0.20923, 0.21398, 0.21175]

In [ ]:
# test=pd.read_csv("../input/optiver-realized-volatility-prediction/test.csv")
# # a=test_predictions_nn*0.5+predictions_lgb2*0.3 + 0.2* preds1
# # b=test_predictions_nn1*0.5+predictions_lgb1*0.3 + 0.2*preds1

# # test[target_name] = W[0] * preds_illi + W[1] * predictions_lgb1 + W[2] * predictions_lgb2 + W[3] * test_predictions_nn + W[4] * test_predictions_nn1

# test[target_name] = W[0] * preds_illi + W[1] * predictions_lgb2 

# display(test[['row_id', target_name]].head(3))
# test[['row_id', target_name]].to_csv('submission.csv',index = False)
# #test[['row_id', target_name]].to_csv('submission.csv',index = False)
# #kmeans N=5 [0.2101, 0.21399, 0.20923, 0.21398, 0.21175]

# Ensemble Weight Optimization

In [ ]:
# oof_lgb_illi = pickle_load("/kaggle/input/optiver-models/oof_lgb_illi.pkl")

# print(oof_lgb_illi.shape)
# print(oof_lgb1.shape)
# print(oof_lgb2.shape)
# print(oof_nn1.shape)
# print(oof_nn2.shape)

# y_target = train_df.target.values
# print(y_target.shape)

## Arit

In [ ]:
# import sys
# def minimize_arit(W):
#     ypred = W[0] * oof_lgb_illi + W[1] * oof_lgb1 + W[2] * oof_lgb2 #+ W[3] * oof_nn1 + W[4] * oof_nn2
#     return rmspe(y_target, ypred )


# constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
# bounds =((0,1),(0,1),(0,1),
# #          (0.01,1),(0.01,1)
#         )

# numwt = 3

# W0 = minimize(minimize_arit, 
#               [1./numwt]*numwt, 
#               options={
# #                   'gtol': 1e-6, 
#                   'disp': True}, 
#               constraints=constraints,
#               bounds=bounds
#              ).x
# print('Weights arit:',W0)

In [ ]:
# import sys
# def minimize_arit(W):
#     ypred = W[0] * oof_nn1 + W[1] * oof_nn2
#     return rmspe(y_target, ypred )


# constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
# bounds =((0,1),(0,1),
# #          (0,1),
# #          (0.01,1),(0.01,1)
#         )

# numwt = 2

# W0 = minimize(minimize_arit, 
#               [1./numwt]*numwt, 
#               options={
# #                   'gtol': 1e-6, 
#                   'disp': True}, 
#               constraints=constraints,
#               bounds=bounds
#              ).x
# print('Weights arit:',W0)

In [ ]:
# W_lgb = [0.80251468, 0., 0.19748532]

# W_nn = [0.5, 0.5]

# preds_lgb = W_lgb[0] * oof_lgb_illi + W_lgb[1] * oof_lgb1 + W_lgb[2] * oof_lgb2
# preds_nn = W_nn[0] * oof_nn1 + W_nn[1] * oof_nn2

# import sys
# def minimize_arit(W):
#     ypred = W[0] * preds_lgb + W[1] * preds_nn
#     return rmspe(y_target, ypred )


# constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
# bounds =((0,1),(0,1),
# #          (0,1),
# #          (0.01,1),(0.01,1)
#         )

# numwt = 2

# W0 = minimize(minimize_arit, 
#               [1./numwt]*numwt, 
#               options={
# #                   'gtol': 1e-6, 
#                   'disp': True}, 
#               constraints=constraints,
#               bounds=bounds
#              ).x
# print('Weights arit:',W0)


## Pow

In [ ]:
# def signed_power(var, p=2):
#     return np.sign(var) * np.abs(var)**p

# def minimize_geom(W):
#     ypred = signed_power(oof_lgb_illi, W[0]) * signed_power(oof_lgb1, W[1]) * signed_power(oof_lgb2, W[2]) * signed_power(oof_nn1, W[3]) * signed_power(oof_nn2, W[4])
#     return rmspe(y_target, ypred)

# W1 = minimize(minimize_geom, [1./5]*5, options={'gtol': 1e-6, 'disp': True}).x

# print('weights geom:',W1)

In [ ]:
# submit_df.to_csv('submission.csv',index = False)